In [6]:
import sys
from pathlib import Path

src_path = Path.cwd() / 'src'
sys.path.append(str(src_path))

In [7]:
from gloomhaven_agent.document_processing.MyPDFReader import MyPDFReader
from Classes.DocumentProcessing.Chunker import Chunker
from Classes.Retrieval.EmbeddingStore import EmbeddingStore
from Classes.Retrieval.Retriever import Retriever
from Classes.GloomHavenAgent.ModelBuilder import ModelBuilder
from Classes.RAG.WebRAGEngine import WebRAGEngine
from Classes.RAG.LocalRAGEngine import LocalRAGEngine
from Classes.GloomHavenAgent.MyAgent import MyAgent
from Classes.Tools.ParamsLoader import ParamsLoader
from Classes.GloomHavenAgent.PredictionGenerator import PredictionGenerator

ModuleNotFoundError: No module named 'document_processing'

In [3]:
import sys
from pathlib import Path

src_path = Path.cwd() / "src"
sys.path.append(str(src_path))

In [1]:
import gloomhaven_agent

ModuleNotFoundError: No module named 'gloomhaven_agent'

In [ ]:
params = ParamsLoader.load_params("utils/params.yaml")
params

{'llm_model_name': 'Qwen/Qwen3-1.7B',
 'embedder_model_name': 'Qwen/Qwen3-Embedding-0.6B',
 'pdf_name': 'rulebook.pdf',
 'Chunker': {'chunk_size': 1000, 'overlap': 200},
 'MyAgent': {'retrieval_threshold': 0.4, 'local_top_k': 5, 'web_top_k': 5}}

In [ ]:
embed_model = ModelBuilder.build_embedder(params['embedder_model_name'])
mypdfreader = MyPDFReader(f'data/{params["pdf_name"]}')
chunker = Chunker(mypdfreader, **params['Chunker'])
embedding_store = EmbeddingStore(chunker, embed_model)
retriever = Retriever(embedding_store, embed_model)
model = ModelBuilder.build_llm(params['llm_model_name'])
local_rag_engine = LocalRAGEngine(retriever, model)
web_rag_engine = WebRAGEngine(model)
agent = MyAgent(local_rag_engine, web_rag_engine, **params['MyAgent'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


In [ ]:
pred_generator = PredictionGenerator(agent)
pred_generator.run('data/eval_dataset.json', 'data/eval_predictions.json')

[{'id': 1,
  'question': 'We attacked a monster and reduced it to 0 HP. Then we applied retaliate damage to the attacker. Did we play this correctly?',
  'expected_was_played_correctly': False,
  'expected_category': 'Combat',
  'prediction': {'explanation': 'The user did not play the retaliate correctly.',
   'was_played_correctly': False,
   'category': 'Combat',
   'source_type': 'rulebook',
   'sources': [{'page_number': 20, 'chunk_number': 3, 'score': 0.607},
    {'page_number': 26, 'chunk_number': 2, 'score': 0.595},
    {'page_number': 26, 'chunk_number': 3, 'score': 0.586},
    {'page_number': 4, 'chunk_number': 2, 'score': 0.581},
    {'page_number': 28, 'chunk_number': 4, 'score': 0.578}],
   'best_retrieval_score': 0.607}},
 {'id': 2,
  'question': 'A character played the top action of one card and the bottom action of another card on the same turn. Did we play this correctly?',
  'expected_was_played_correctly': True,
  'expected_category': 'Character',
  'prediction': {'ex

In [4]:
answer = agent.answer('What are the most frequently debated Gloomhaven rules according to the online community?')

In [5]:
answer

{'explanation': 'The user is asking about the most frequently debated Gloomhaven rules based on online community discussions.',
 'was_played_correctly': True,
 'category': 'Scenario',
 'source_type': 'web',
 'sources': [{'title': "I'm making a visual(ish) comparison among the rules we have ...",
   'href': 'https://www.facebook.com/groups/646259685567420/posts/1758029021057142/',
   'body': "Dec 30, 2021 ... Jason Herdman I'd only played a couple scenarios in Gloomhaven before hopping into Jaws of the Lion, but I'd read the base game rules so much we ..."},
  {'title': 'Limits of strategy discussions between players - Gloomhaven - Reddit',
   'href': 'https://www.reddit.com/r/Gloomhaven/comments/1ijs4fv/limits_of_strategy_discussions_between_players/',
   'body': 'Feb 7, 2025 ... There are no limits to strategy discussion. Only card names and exact initiative numbers are off-limits. Play how you want to play,, with a ...'},
  {'title': 'How to Play Gloomhaven Part 1 (The Basics) - YouT